# 03 — Gold Transformation (Star Schema Build)Driver notebook for Chapter 5. Reads `silver_job_postings` and builds the Goldstar schema: four dimensions, one fact table, one bridge table.Order matters — the fact table resolves surrogate keys by joining against thedimensions, so every dimension must be current before `build_fact_and_bridge`runs:1. `dim_date` — generated once for a wide range, no incremental logic2. `dim_skill` — rebuilt from the curated taxonomy3. `dim_location` — Type 1 upsert from Silver4. `dim_company` — **SCD Type 2** upsert from Silver5. `fact_job_postings` + `bridge_job_skill` — built, then MERGEdAll transformation logic lives in `transformation/dim_builders.py` and`transformation/silver_to_gold.py`. This notebook only orchestrates andinspects — no business logic below.

In [ ]:
from datetime import datefrom delta.tables import DeltaTablefrom pyspark.sql import SparkSessionfrom pyspark.sql import functions as Ffrom utils.config_loader import load_configfrom utils.logger import setup_logging, get_loggerfrom transformation.dim_builders import (    generate_dim_date,    upsert_dim_company_scd2,    upsert_dim_location,    upsert_dim_skill,)from transformation.silver_to_gold import (    build_fact_and_bridge,    merge_fact_table,    optimize_fact_table,    vacuum_fact_table,)setup_logging()logger = get_logger(__name__)spark = SparkSession.builder.appName("gold_transformation").getOrCreate()app_config = load_config()

> **Running this locally instead of in Fabric:** a Fabric notebook session has> Delta wired into the Spark catalog already. A plain local `getOrCreate()` does> not, and every `DeltaTable` call below fails with `Delta is not enabled`.> Build the session like this instead:>> ```python> from delta import configure_spark_with_delta_pip>> builder = (>     SparkSession.builder.appName("gold_transformation")>     .master("local[2]")>     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")>     .config("spark.sql.catalog.spark_catalog",>             "org.apache.spark.sql.delta.catalog.DeltaCatalog")> )> spark = configure_spark_with_delta_pip(builder).getOrCreate()> ```>> The first run downloads the Delta jars, so it needs network access.

## Gold table pathsSame `Tables/` convention as notebooks 01 and 02 — in Fabric these resolveinside the Lakehouse; locally they are plain directories relative to the reporoot.

In [ ]:
SILVER_PATH = "Tables/silver_job_postings"DIM_DATE_PATH = "Tables/dim_date"DIM_SKILL_PATH = "Tables/dim_skill"DIM_LOCATION_PATH = "Tables/dim_location"DIM_COMPANY_PATH = "Tables/dim_company"FACT_PATH = "Tables/fact_job_postings"BRIDGE_PATH = "Tables/bridge_job_skill"# The run date stamps SCD2 effective dates. Parameterised rather than hardcoded# to date.today() so a backfill can replay a historical day and still produce# correct effective_start_date / effective_end_date values.RUN_DATE = date.today()print("Run date:", RUN_DATE)

## Step 1 — `dim_date`Static dimension: generated once over a wide range rather than incrementally.Regenerating is cheap, but there is no reason to rewrite it every run, so thiscell is a no-op once the table exists.

In [ ]:
if DeltaTable.isDeltaTable(spark, DIM_DATE_PATH):    print("dim_date already exists — skipping generation")else:    dim_date_df = generate_dim_date(spark, date(2020, 1, 1), date(2030, 12, 31))    dim_date_df.write.format("delta").mode("overwrite").save(DIM_DATE_PATH)    print("Generated dim_date rows:", dim_date_df.count())spark.read.format("delta").load(DIM_DATE_PATH).orderBy("date_key").show(5)

## Step 2 — `dim_skill`Rebuilt in full from `SKILL_TAXONOMY`. The taxonomy is a curated constant, notdata arriving from upstream, so a full overwrite is simpler and safer than anupsert — a skill removed from the taxonomy should disappear from the dimension.

In [ ]:
upsert_dim_skill(spark, DIM_SKILL_PATH)dim_skill = spark.read.format("delta").load(DIM_SKILL_PATH)print("dim_skill rows:", dim_skill.count())dim_skill.groupBy("skill_category").count().orderBy(F.desc("count")).show()

## Step 3 — Read SilverThe single input to the whole Gold build. Everything downstream derives fromthis DataFrame.

In [ ]:
silver_df = spark.read.format("delta").load(SILVER_PATH)print("Silver rows:", silver_df.count())print("Distinct companies:", silver_df.select("company").distinct().count())print("Distinct countries:", silver_df.select("canonical_country").distinct().count())silver_df.printSchema()

## Step 4 — `dim_location` (SCD Type 1)New `(canonical_country, region)` pairs are inserted; existing ones are leftalone. No history is kept — a country's canonical region changing is a datacorrection, not a business fact anyone will ask to see historically.

In [ ]:
upsert_dim_location(spark, silver_df, DIM_LOCATION_PATH)spark.read.format("delta").load(DIM_LOCATION_PATH).show(20, truncate=False)

## Step 5 — `dim_company` (SCD Type 2)The only Type 2 dimension in the model. `size_bucket` is derived from how manypostings a company has and genuinely evolves over the observation window —"has this company's hiring scale changed" is a real question the project'sobjectives imply.On each run, per company:- **new company** → insert a current row- **`size_bucket` changed** → close the old row (`is_current = false`,  `effective_end_date = RUN_DATE`) and insert a new current row with a new  surrogate key- **unchanged** → leave the existing row untouched

In [ ]:
upsert_dim_company_scd2(    spark=spark,    silver_df=silver_df,    table_path=DIM_COMPANY_PATH,    run_date=RUN_DATE,)dim_company = spark.read.format("delta").load(DIM_COMPANY_PATH)print("dim_company total rows:", dim_company.count())print("dim_company current rows:", dim_company.filter(F.col("is_current")).count())dim_company.orderBy("company_natural_key", "effective_start_date").show(20, truncate=False)

## Step 6 — `fact_job_postings` + `bridge_job_skill``build_fact_and_bridge` resolves every Silver row against the dimensions above(company via the **current** SCD2 row, location, date) and extracts skills fromthe description to produce the many-to-many bridge.The fact load is a MERGE on `source_job_id`, not an append. Silver reprocessingmeans the same posting can legitimately reach Gold more than once — a lateJooble update, or a DQ rule change releasing a previously quarantined row. Anappend-only fact would double-count that posting in every KPI.

In [ ]:
fact_df, bridge_df = build_fact_and_bridge(    spark=spark,    silver_df=silver_df,    dim_company_path=DIM_COMPANY_PATH,    dim_location_path=DIM_LOCATION_PATH,    dim_date_path=DIM_DATE_PATH,    dim_skill_path=DIM_SKILL_PATH,)fact_df.cache()print("Fact rows staged:", fact_df.count())print("Bridge rows staged:", bridge_df.count())# Unresolved keys mean a dimension is stale or a join key is dirty. Check here# rather than discovering NULL keys in Power BI three chapters later.fact_df.select(    F.sum(F.col("company_key").isNull().cast("int")).alias("null_company_key"),    F.sum(F.col("location_key").isNull().cast("int")).alias("null_location_key"),    F.sum(F.col("date_key").isNull().cast("int")).alias("null_date_key"),).show()

In [ ]:
merge_fact_table(spark, fact_df, FACT_PATH)

`silver_to_gold.py` has no bridge helper — the bridge is a pure link table witha composite key, so an insert-if-absent MERGE keeps it idempotent with no updatebranch at all. Worth promoting into the module if Chapter 6's orchestrator endsup calling it too.

In [ ]:
if not DeltaTable.isDeltaTable(spark, BRIDGE_PATH):    bridge_df.write.format("delta").save(BRIDGE_PATH)    print("Initialized bridge_job_skill")else:    (        DeltaTable.forPath(spark, BRIDGE_PATH).alias("target")        .merge(            bridge_df.alias("source"),            "target.source_job_id = source.source_job_id "            "AND target.skill_key = source.skill_key",        )        .whenNotMatchedInsertAll()        .execute()    )    print("Merged into bridge_job_skill")

## Verify the star schemaRow counts across every Gold table, then the joins the Power BI semantic modelin Chapter 7 will rely on. If these return sensible results, the surrogate keysresolved correctly end to end.

In [ ]:
for name, path in [    ("dim_date", DIM_DATE_PATH),    ("dim_skill", DIM_SKILL_PATH),    ("dim_location", DIM_LOCATION_PATH),    ("dim_company", DIM_COMPANY_PATH),    ("fact_job_postings", FACT_PATH),    ("bridge_job_skill", BRIDGE_PATH),]:    count = spark.read.format("delta").load(path).count()    print(f"{name:22} {count:>8} rows")

In [ ]:
fact = spark.read.format("delta").load(FACT_PATH)bridge = spark.read.format("delta").load(BRIDGE_PATH)dim_skill = spark.read.format("delta").load(DIM_SKILL_PATH)# Most in-demand skills — the headline question the whole project exists to answer.(    bridge.join(dim_skill, "skill_key")    .groupBy("skill_name", "skill_category")    .count()    .orderBy(F.desc("count"))    .show(15, truncate=False))

In [ ]:
dim_company = spark.read.format("delta").load(DIM_COMPANY_PATH)dim_location = spark.read.format("delta").load(DIM_LOCATION_PATH)# Salary by country and company size — exercises three dimension joins at once.(    fact.join(dim_company.filter(F.col("is_current")), "company_key")    .join(dim_location, "location_key")    .filter(F.col("salary_min").isNotNull())    .groupBy("canonical_country", "size_bucket")    .agg(        F.count("*").alias("postings"),        F.round(F.avg("salary_min")).alias("avg_salary_min"),        F.round(F.avg("salary_max")).alias("avg_salary_max"),    )    .orderBy(F.desc("postings"))    .show(20, truncate=False))

## Verify SCD Type 2 actually workedOnly meaningful from the **second** run onwards — the first run has nothing toclose out. After a run where some company's posting count crosses a bucketboundary, that company should show two rows: one closed, one current.

In [ ]:
history = (    dim_company    .groupBy("company_natural_key")    .agg(F.count("*").alias("versions"))    .filter(F.col("versions") > 1))print("Companies with SCD2 history:", history.count())(    dim_company.join(history, "company_natural_key")    .select(        "company_natural_key", "company_key", "size_bucket",        "effective_start_date", "effective_end_date", "is_current",    )    .orderBy("company_natural_key", "effective_start_date")    .show(20, truncate=False))

In [ ]:
# Invariant: exactly one current row per company. A violation means the MERGE's# close-out branch failed, and every fact join would fan out into duplicates.violations = (    dim_company.filter(F.col("is_current"))    .groupBy("company_natural_key").count()    .filter(F.col("count") > 1))assert violations.count() == 0, "SCD2 invariant violated: multiple current rows"print("SCD2 invariant holds: exactly one current row per company")

## Delta maintenance — OPTIMIZE, Z-ORDER, VACUUMNot part of the per-run pipeline. Trickling small writes into a partitioned facttable produces the small-file problem; `OPTIMIZE` compacts them and Z-ORDERco-locates the columns the Power BI filters hit hardest. `VACUUM` then dropsfiles no longer referenced by any retained version.Both are expensive and belong on a weekly schedule — that scheduling isChapter 6's job. Run them manually here to see what they do.**`VACUUM` is destructive:** it deletes the underlying files time travel dependson. The 168-hour (7 day) default is the retention floor Delta enforces for areason — lowering it can break in-flight readers.

In [ ]:
optimize_fact_table(spark, FACT_PATH)

In [ ]:
# Leaves the default 7-day retention intact, so the time travel below still works.vacuum_fact_table(spark, FACT_PATH, retention_hours=168)

## Time travelEvery write to a Delta table creates a new version, and the transaction log keepsold versions addressable. That makes "what changed since yesterday's load"answerable without maintaining a snapshot table.

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{FACT_PATH}`").select(    "version", "timestamp", "operation", "operationMetrics").show(10, truncate=False)

In [ ]:
# Compare the current table against its first version. versionAsOf rather than a# hardcoded timestampAsOf date, so this cell works on a fresh table — a date# older than the table's first commit raises an error instead of returning rows.today_df = spark.read.format("delta").load(FACT_PATH)first_df = spark.read.format("delta").option("versionAsOf", 0).load(FACT_PATH)print("Row count now:", today_df.count(), "| at version 0:", first_df.count())# The timestamp form, for reference — swap in a date the table actually spans:# yesterday_df = (#     spark.read.format("delta")#     .option("timestampAsOf", "2026-07-14")#     .load(FACT_PATH)# )

## Schema evolution`mergeSchema` lets a new column appear simply by writing a DataFrame that hasit — no manual DDL. Declaring the change explicitly is still better practice: itputs the intent in the transaction log and in code review, rather than leaving acolumn to materialise silently on some future run.

In [ ]:
spark.sql(f"ALTER TABLE delta.`{FACT_PATH}` ADD COLUMNS (contract_type STRING)")spark.read.format("delta").load(FACT_PATH).printSchema()

In [ ]:
fact_df.unpersist()spark.stop()